# Обучение модели fraud detection


In [ ]:
from __future__ import annotations

import json
import os
from datetime import datetime, timezone
from pathlib import Path

import catboost
import joblib
import pandas as pd
import sklearn
from catboost import CatBoostClassifier
from sklearn.metrics import average_precision_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split

from src.preprocessing import (
    CATEGORICAL_COLUMNS, MODEL_CATEGORICAL_COLUMNS, MODEL_FEATURE_COLUMNS,
    NUMERIC_COLUMNS, TARGET, fit_preprocessor, transform,
)

RANDOM_SEED = 42
THRESHOLD = float(os.getenv("MODEL_THRESHOLD", "0.98"))
ITERATIONS = int(os.getenv("CATBOOST_ITERATIONS", "500"))


DATA_PATH = Path(os.getenv("DATA_DIR")) / "train.csv"
MODEL_VERSION = os.getenv("MODEL_VERSION", datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S"))
ARTIFACT_DIR = Path(os.getenv("ARTIFACT_DIR")) / MODEL_VERSION
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Данные: {DATA_PATH}\nАртефакты: {ARTIFACT_DIR}")

Данные: /app/data/train.csv
Артефакты: /app/artifacts/20261009-140051


## Препроцессинг


In [3]:
raw = pd.read_csv(DATA_PATH)
if TARGET not in raw.columns:
    raise ValueError(f"В {DATA_PATH} отсутствует target-колонка {TARGET!r}")
if raw[TARGET].isna().any():
    raise ValueError("В target есть пропуски")

train_raw, valid_raw = train_test_split(raw, test_size=0.2, random_state=RANDOM_SEED, stratify=raw[TARGET])
preprocessor = fit_preprocessor(train_raw)
X_train, X_valid = transform(train_raw, preprocessor), transform(valid_raw, preprocessor)
y_train, y_valid = train_raw[TARGET].astype(int), valid_raw[TARGET].astype(int)
print(f"train={X_train.shape}, validation={X_valid.shape}, fraud rate={y_train.mean():.4%}")
X_train.head()

train=(629144, 12), validation=(157287, 12), fraud rate=0.5727%


,gender,merch,cat_id,one_city,us_state,jobs,hour,month,day_of_week,amount_log,population_city_log,distance_km_log
433955,M,fraud_Cummerata-Hilpert,entertainment,Brooklin,ME,"Loss adjuster, chartered",18,1,1,4.338728,6.715383,4.780764
259607,F,fraud_Jast Ltd,shopping_net,Lakeland,FL,Clothing/textile technologist,19,2,6,3.943909,12.377009,4.420576
556819,F,fraud_McDermott-Weimann,grocery_pos,Philadelphia,PA,Structural engineer,10,2,0,5.293506,14.238296,4.682014
652395,F,fraud_Breitenberg-Hermiston,kids_pets,New York City,NY,Herbalist,15,7,2,3.371769,14.271280,4.745329
671918,F,"fraud_Swaniawski, Lowe and Robel",shopping_pos,Gardiner,OR,"Engineer, maintenance",2,4,2,2.074429,5.564520,4.766594


## Обучение и оценка

In [4]:
model = CatBoostClassifier(
    iterations=ITERATIONS, depth=8, learning_rate=0.08, loss_function="Logloss",
    eval_metric="AUC", auto_class_weights="Balanced", random_seed=RANDOM_SEED,
    allow_writing_files=False, verbose=50,
)

model.fit(X_train, y_train, cat_features=MODEL_CATEGORICAL_COLUMNS, eval_set=(X_valid, y_valid), early_stopping_rounds=50)
valid_scores = model.predict_proba(X_valid)[:, 1]
valid_predictions = (valid_scores >= THRESHOLD).astype(int)

metrics = {
    "roc_auc": float(roc_auc_score(y_valid, valid_scores)),
    "average_precision": float(average_precision_score(y_valid, valid_scores)),
    "precision_at_threshold": float(precision_score(y_valid, valid_predictions, zero_division=0)),
    "recall_at_threshold": float(recall_score(y_valid, valid_predictions, zero_division=0)),
    "f1_at_threshold": float(f1_score(y_valid, valid_predictions, zero_division=0)),
    "threshold": THRESHOLD, "train_rows": int(len(train_raw)),
    "validation_rows": int(len(valid_raw)), "positive_rate_train": float(y_train.mean()),
    "positive_rate_validation": float(y_valid.mean()), "best_iteration": int(model.get_best_iteration()),
}
pd.Series(metrics)

0:	test: 0.9390210	best: 0.9390210 (0)	total: 274ms	remaining: 2m 16s
50:	test: 0.9990737	best: 0.9990737 (50)	total: 12.7s	remaining: 1m 51s
100:	test: 0.9992226	best: 0.9992226 (100)	total: 24.3s	remaining: 1m 35s
150:	test: 0.9993839	best: 0.9993839 (150)	total: 35.3s	remaining: 1m 21s
200:	test: 0.9995734	best: 0.9995860 (197)	total: 47.8s	remaining: 1m 11s
250:	test: 0.9996733	best: 0.9996733 (250)	total: 1m	remaining: 1m
300:	test: 0.9996967	best: 0.9996967 (300)	total: 1m 15s	remaining: 50s
350:	test: 0.9997408	best: 0.9997410 (347)	total: 1m 28s	remaining: 37.8s
400:	test: 0.9997600	best: 0.9997604 (391)	total: 1m 42s	remaining: 25.2s
450:	test: 0.9997757	best: 0.9997762 (446)	total: 1m 55s	remaining: 12.6s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.9997762374
bestIteration = 446

Shrink model to first 447 iterations.


roc_auc                          0.999776
average_precision                0.983414
precision_at_threshold           0.983871
recall_at_threshold              0.880133
f1_at_threshold                  0.929115
threshold                        0.980000
train_rows                  629144.000000
validation_rows             157287.000000
positive_rate_train              0.005727
positive_rate_validation         0.005728
best_iteration                 446.000000
dtype: float64

## Сохранение артефактов

In [5]:
metadata = {
    "model_version": MODEL_VERSION, "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "model_type": "CatBoostClassifier", "threshold": THRESHOLD, "random_seed": RANDOM_SEED,
    "target": TARGET, "training_data": str(DATA_PATH.resolve()),
    "model_file": "model.cbm",
    "preprocessor_file": "preprocessor.joblib", 
    "feature_schema_file": "feature_schema.json",
}
feature_schema = {
    "raw_required_columns": {"transaction_time": "datetime string",
        **{column: "string" for column in CATEGORICAL_COLUMNS},
        **{column: "number" for column in NUMERIC_COLUMNS}},
    "target_column": TARGET, "model_feature_columns": MODEL_FEATURE_COLUMNS,
    "categorical_feature_columns": MODEL_CATEGORICAL_COLUMNS,
}

model.save_model(ARTIFACT_DIR / "model.cbm")
joblib.dump(preprocessor, ARTIFACT_DIR / "preprocessor.joblib")

for filename, payload in [("metadata.json", metadata), ("feature_schema.json", feature_schema), ("metrics.json", metrics)]:
    with (ARTIFACT_DIR / filename).open("w", encoding="utf-8") as file:
        json.dump(payload, file, ensure_ascii=False, indent=2)
        
print(f"Сохранено в {ARTIFACT_DIR}: {sorted(path.name for path in ARTIFACT_DIR.iterdir())}")

Сохранено в /app/artifacts/20261009-140051: ['feature_schema.json', 'metadata.json', 'metrics.json', 'model.cbm', 'preprocessor.joblib']


## Smoke test инференса

Проверяем артефакты после повторной загрузки с диска. Этот же порядок действий используется сервисом инференса.

In [ ]:
loaded_preprocessor = joblib.load(ARTIFACT_DIR / "preprocessor.joblib")
loaded_model = CatBoostClassifier()
loaded_model.load_model(ARTIFACT_DIR / "model.cbm")
with (ARTIFACT_DIR / "metadata.json").open(encoding="utf-8") as file:
    loaded_metadata = json.load(file)
    
sample = valid_raw.iloc[:5].drop(columns=[TARGET])
sample_features = transform(sample, loaded_preprocessor)
sample_scores = loaded_model.predict_proba(sample_features)[:, 1]
pd.DataFrame({"score": sample_scores, "fraud_flag": (sample_scores >= loaded_metadata["threshold"]).astype(int)}, index=sample.index)

,score,fraud_flag
201970,9.587372e-03,0
487081,2.191197e-07,0
446848,5.482915e-06,0
719069,1.192589e-03,0
37753,2.826584e-06,0
